# One table for the map

The explorer only reads `data/processed/district_panel.csv`.
This notebook stacks two kinds of rows:

1. **crop** — yearbook yield / area, plus rainfall for that season
2. **rain_only** — rainfall still filled, crop left blank, plus a sentence for why yield is grey

I do not invent yield. If the yearbook has no row, the map stays grey and the tooltip explains it.

Run `crop_clean.ipynb` and `imd_clean.ipynb` first.

In [1]:
from pathlib import Path

import pandas as pd

crop_path = Path("data/processed/crop_clean.csv")
rain_path = Path("data/processed/rainfall_district_seasonal.csv")
out_path = Path("data/processed/district_panel.csv")

if not crop_path.exists():
    raise SystemExit("Run crop_clean.ipynb first.")
if not rain_path.exists():
    raise SystemExit("Run imd_clean.ipynb first.")

crop = pd.read_csv(crop_path)
rain = pd.read_csv(rain_path)
print("crop", crop.shape, "rain", rain.shape)
crop.head(2)

crop (424083, 9) rain (62640, 10)


,district_key,district,state,year,season,crop,area_ha,production_tonnes,yield_t_ha
0,adilabad,Adilabad,Telangana,1997-1998,Kharif,Arhar/Tur,32200.0,1100.0,0.034
1,adilabad,Adilabad,Telangana,1997-1998,Kharif,Castor Seed,2600.0,700.0,0.269


In [2]:
def rain_window(season):
    s = str(season or "").strip().lower()
    if s == "kharif":
        return "Kharif"
    if s == "rabi":
        return "Rabi"
    return "Annual"


print(crop["season"].value_counts())
print(rain["season"].value_counts())

season
Kharif        150973
Rabi          112349
Whole Year     76186
Total          44249
Summer         24328
Winter          8800
Autumn          7198
Name: count, dtype: int64
season
Annual    21141
Kharif    21141
Rabi      20358
Name: count, dtype: int64


##  Join rainfall onto each crop row

In [3]:
rain = rain.rename(columns={"key": "district_key"})
rain["district_key"] = rain["district_key"].astype(str)
crop["district_key"] = crop["district_key"].astype(str)

# one rain number per district-year-season (Kharif / Rabi / Annual)
rain_keep = rain[
    ["district_key", "year", "season", "state", "district",
     "rain_mm", "rain_normal_mm", "rain_anom_pct"]
].drop_duplicates(["district_key", "year", "season"])

crop = crop.copy()
crop["rain_join"] = crop["season"].map(rain_window)

crop = crop.merge(
    rain_keep.rename(columns={
        "season": "rain_join",
        "state": "rain_state",
        "district": "rain_district",
    }),
    on=["district_key", "year", "rain_join"],
    how="left",
)

# if crop state/district was blank, take the rain table's names
crop["state"] = crop["state"].where(crop["state"].astype(str).str.len() > 0, crop["rain_state"])
crop["district"] = crop["district"].where(crop["district"].astype(str).str.len() > 0, crop["rain_district"])

crop["row_type"] = "crop"
crop["crop_missing_reason"] = ""
print("crop rows after join", len(crop))
crop.head(2)

crop rows after join 424083


,district_key,district,state,year,season,crop,area_ha,production_tonnes,yield_t_ha,rain_join,rain_state,rain_district,rain_mm,rain_normal_mm,rain_anom_pct,row_type,crop_missing_reason
0,adilabad,Adilabad,Telangana,1997-1998,Kharif,Arhar/Tur,32200.0,1100.0,0.034,Kharif,TELANGANA,ADILABAD,754.21,1002.64,-24.78,crop,
1,adilabad,Adilabad,Telangana,1997-1998,Kharif,Castor Seed,2600.0,700.0,0.269,Kharif,TELANGANA,ADILABAD,754.21,1002.64,-24.78,crop,


## Rain-only rows (why the map is grey)

Rain is computed on **today’s** polygons for every year. Crop statistics only exist when DES published them.
So I add a second copy of the rain table with `crop` blank. The tooltip reads `crop_missing_reason`.

Typical cases:
- new split (e.g. Tirupati) — on the map, not in old yearbooks
- district exists in the yearbook, but not this year / this season
- Kharif has crops, Total does not — Annual rain is still there

In [4]:
first_year = crop.groupby("district_key")["year"].min().to_dict()
crop_keys = set(crop["district_key"])
has_year = set(zip(crop["district_key"], crop["year"]))
has_season = set(zip(crop["district_key"], crop["year"], crop["season"]))


def why_no_crop(key, year, season):
    if key not in crop_keys:
        return (
            "No crop statistics for this district. It is on today's map but never "
            "appears in the crop yearbook — usually a new split, a truncated map "
            "label we could not match, or a polygon with no name."
        )
    if year < first_year.get(key, year):
        return (
            f"No crop yearbook row until {first_year[key]}. Current boundaries are used on "
            "the map, so this district can exist here before it exists in the table."
        )
    if (key, year) not in has_year:
        return f"This district is in the yearbook, but not for {year}."
    if (key, year, season) in has_season:
        return (
            f"Crops were reported in {season} {year}, but not necessarily the crop "
            "on the menu. Pick a crop that was grown here. Rainfall is still filled."
        )
    if season == "Annual":
        return (
            "No Total / Whole Year / Summer crop row for this district-year "
            "(those seasons borrow Annual rainfall). Try Kharif if you want yield."
        )
    return (
        f"{season} is not a crop-reporting season for this district in {year}. "
        "Rain still uses Kharif / Rabi / Annual windows. Try Kharif if Total is empty."
    )

In [5]:
rain_only = rain_keep.copy()
rain_only["crop"] = ""
rain_only["area_ha"] = pd.NA
rain_only["production_tonnes"] = pd.NA
rain_only["yield_t_ha"] = pd.NA
rain_only["row_type"] = "rain_only"
rain_only["crop_missing_reason"] = [
    why_no_crop(r.district_key, r.year, r.season)
    for r in rain_only.itertuples(index=False)
]
print("rain-only rows", len(rain_only))

rain-only rows 62640


##  Same columns, one CSV

In [6]:
cols = [
    "state", "district", "district_key", "year", "season", "crop",
    "area_ha", "production_tonnes", "yield_t_ha",
    "rain_mm", "rain_normal_mm", "rain_anom_pct",
    "row_type", "crop_missing_reason",
]

panel = pd.concat([crop[cols], rain_only[cols]], ignore_index=True)
panel["crop_missing_reason"] = panel["crop_missing_reason"].fillna("")
panel.to_csv(out_path, index=False)

print("crop rows", (panel["row_type"] == "crop").sum())
print("rain-only rows", (panel["row_type"] == "rain_only").sum())
print("wrote", len(panel), "->", out_path)
panel.head()

C:\Users\likit\AppData\Local\Temp\ipykernel_21308\4261360158.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  panel = pd.concat([crop[cols], rain_only[cols]], ignore_index=True)


crop rows 424083
rain-only rows 62640
wrote 486723 -> data\processed\district_panel.csv


,state,district,district_key,year,season,crop,area_ha,production_tonnes,yield_t_ha,rain_mm,rain_normal_mm,rain_anom_pct,row_type,crop_missing_reason
0,Telangana,Adilabad,adilabad,1997-1998,Kharif,Arhar/Tur,32200.0,1100.0,0.034,754.21,1002.64,-24.78,crop,
1,Telangana,Adilabad,adilabad,1997-1998,Kharif,Castor Seed,2600.0,700.0,0.269,754.21,1002.64,-24.78,crop,
2,Telangana,Adilabad,adilabad,1997-1998,Kharif,Cotton(Lint),144900.0,66500.0,0.459,754.21,1002.64,-24.78,crop,
3,Telangana,Adilabad,adilabad,1997-1998,Kharif,Dry Chillies,5500.0,3100.0,0.564,754.21,1002.64,-24.78,crop,
4,Telangana,Adilabad,adilabad,1997-1998,Kharif,Jowar,58200.0,31500.0,0.541,754.21,1002.64,-24.78,crop,
